In [5]:
import pandas as pd
import numpy as np
import os
from tqdm.notebook import tqdm

# --- 1. Configuration: Define File Paths ---

# INPUT directory (Read-Only on Kaggle)
INPUT_DIR = '/kaggle/input/m5-forecasting-accuracy'
SALES_FILE = os.path.join(INPUT_DIR, 'sales_train_validation.csv')
CALENDAR_FILE = os.path.join(INPUT_DIR, 'calendar.csv')
PRICES_FILE = os.path.join(INPUT_DIR, 'sell_prices.csv')

# OUTPUT directory (Writable on Kaggle)
OUTPUT_DIR = '/kaggle/working'
LEVEL_11_SALES_FILE = os.path.join(OUTPUT_DIR, 'sales_train_level_11.csv')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv')


# --- 2. Utility Functions ---

def downcast_df(df):
    """Reduces memory usage by changing column dtypes."""
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    return df

def calculate_ts_features(series):
    """
    Calculates Average Inter-Demand Interval (ADI) and Squared Coefficient
    of Variation (CV²) for a time series (Phase 1, Step 3).
    """
    # Filter for non-zero demand values
    demand = series[series > 0]
    
    total_periods = len(series)
    non_zero_periods = len(demand)
    
    # ADI (Intermittency)
    if non_zero_periods == 0:
        adi = total_periods
        cv2 = 0.0
    else:
        adi = total_periods / non_zero_periods
        
        # CV² (Variability of demand size)
        mean_demand = demand.mean()
        std_demand = demand.std(ddof=1)
        
        if mean_demand == 0 or np.isnan(std_demand):
             cv2 = 0.0
        else:
            cv2 = (std_demand / mean_demand) ** 2

    return pd.Series({'ADI': adi, 'CV2': cv2})

# --- 3. Phase 1 Execution ---

# 3a. Load Sales Data
print("1. Loading Sales Data...")
# We only need 'item_id', 'state_id', and the daily sales columns ('d_').
cols_to_read = ['item_id', 'state_id'] + [f'd_{i}' for i in range(1, 1914)] 
sales_df = pd.read_csv(SALES_FILE, usecols=lambda x: x in cols_to_read or x.startswith('d_'))
sales_df = downcast_df(sales_df)
day_cols = [col for col in sales_df.columns if col.startswith('d_')]


# 3b. Aggregation to Product-State (Level 11)
print("2. Aggregating to Product-State (Level 11)...")

# Group by the Level 11 keys and sum the daily sales
level_11_df = sales_df.groupby(['item_id', 'state_id'])[day_cols].sum().reset_index()

# Add a unique ID for convenience
level_11_df['id'] = level_11_df['item_id'].astype(str) + '_' + level_11_df['state_id'].astype(str)

print(f"Aggregation Complete. Number of Level 11 series: {len(level_11_df)}")

# Save the Level 11 aggregated data (for use in simulation.py)
level_11_df.to_csv(LEVEL_11_SALES_FILE, index=False)
print(f"Level 11 data saved to: {LEVEL_11_SALES_FILE}")


# 3c. Calculate ADI and CV² Features (Phase 1, Step 3)
print("3. Calculating ADI and CV² Time Series Features...")

# Apply feature calculation to each time series row
ts_features = level_11_df[day_cols].apply(calculate_ts_features, axis=1)

# Merge features with identifiers
feature_cols = ['id', 'item_id', 'state_id']
features_df = level_11_df[feature_cols].copy()
features_df = pd.concat([features_df, ts_features], axis=1)

# Save the features
features_df.to_csv(FEATURES_FILE, index=False)
print(f"Feature calculation complete. Number of series: {len(features_df)}")
print(f"Time series features (ADI and CV²) saved to: {FEATURES_FILE}")


# 3d. Load and Save Price/Calendar Data (Cleanup and Readiness Check)
print("\n4. Loading and Preparing Calendar and Price Data (Optional save to /kaggle/working)...")
calendar_df = pd.read_csv(CALENDAR_FILE)
prices_df = pd.read_csv(PRICES_FILE)

# Save prices and calendar to a common working directory if needed by later scripts
calendar_df.to_csv(os.path.join(OUTPUT_DIR, 'calendar.csv'), index=False)
prices_df.to_csv(os.path.join(OUTPUT_DIR, 'sell_prices.csv'), index=False)
print("Calendar and Price data saved to writable directory.")

# --- Next Step ---
print("\n--- Next Step (Phase 2, Step 4 & 5) ---")
print("5. Generate Hyperparameter Sets and Run Stock Control Simulations.")
print("The script name for the next step should be: simulation.py")

1. Loading Sales Data...
2. Aggregating to Product-State (Level 11)...
Aggregation Complete. Number of Level 11 series: 9147
Level 11 data saved to: /kaggle/working/sales_train_level_11.csv
3. Calculating ADI and CV² Time Series Features...
Feature calculation complete. Number of series: 9147
Time series features (ADI and CV²) saved to: /kaggle/working/m5_demand_features.csv

4. Loading and Preparing Calendar and Price Data (Optional save to /kaggle/working)...
Calendar and Price data saved to writable directory.

--- Next Step (Phase 2, Step 4 & 5) ---
5. Generate Hyperparameter Sets and Run Stock Control Simulations.
The script name for the next step should be: simulation.py


In [6]:
import os
import pandas as pd
import numpy as np
from scipy.stats import norm
from itertools import product
from tqdm import tqdm

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory (Must be /kaggle/working for writable storage)
OUTPUT_DIR = '/kaggle/working'

# Input files from previous steps
LEVEL_11_SALES_FILE = os.path.join(OUTPUT_DIR, 'sales_train_level_11.csv')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv')

# Output file for ML training (Features + Targets)
# Changed filename to reflect the new hparam set
SIMULATION_OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'ml_training_data_75_hparams.csv')

# Simulation period constants 
# We use the last 365 days of the historical data for the OFF-LINE simulation 
# (Based on the paper's use of a full calendar year for the off-line phase)
TRAIN_DAYS = 1913 # Max day in sales_train_validation.csv
SIM_DAYS = 365
SPLIT_DAY = TRAIN_DAYS - SIM_DAYS
DAY_COLS = [f'd_{i}' for i in range(SPLIT_DAY + 1, TRAIN_DAYS + 1)] # Last 365 days

# --- 2. Hyperparameter Ranges (Phase 2, Step 4) ---

print("!!! PROTOTYPING MODE: Using reduced hyperparameter set (75 sets) !!!")

# --- MODIFICATION: Reduced Hyperparameters ---
# R: Review Period (5 values instead of 30)
R_RANGE = np.linspace(1, 30, 5, dtype=int)
# L: Lead Time (5 values instead of 30)
L_RANGE = np.linspace(1, 30, 5, dtype=int)
# TSL: Target Service Level (3 values instead of 10)
TSL_RANGE = np.array([0.90, 0.95, 0.99])

# Generate all 5 x 5 x 3 = 75 hyperparameter sets (down from 9,000)
POLICY_HPARAMS = list(product(R_RANGE, L_RANGE, TSL_RANGE))
# ----------------------------------------------------

print(f"4. Generating {len(POLICY_HPARAMS)} Inventory Policy Hyperparameter Sets...")

# --- 3. Core MOIC Simulation Functions (Phase 2, Step 5 Logic) ---

def calculate_policy_stats(history):
    """
    Calculates D_hat (Average Demand) and sigma_D (Std Dev) for SS calculation.
    """
    D_hat = history.mean() 
    # Use ddof=1 for sample standard deviation
    sigma_D = history.std(ddof=1) if len(history) > 1 else 1.0
    return D_hat, sigma_D

def run_rss_simulation(ts_sales, R, L, TSL):
    """
    Runs the (R, s, S) lost-sales stock control simulation over the reference period.
    Returns the three Inventory Cost Elements (Targets).
    """
    
    # 1. Pre-calculate Policy Parameters (Safety Stock and S/s levels)
    D_hat, sigma_D = calculate_policy_stats(ts_sales)
    
    # Z-score: Z = Φ⁻¹(TSL) 
    Z = norm.ppf(TSL)
    
    # Safety Stock (SS): SS = Z * σ_D * sqrt(R + L) (Greasley's formula, Eq. 6 in paper)
    SS = Z * sigma_D * np.sqrt(R + L)
    
    # Order Point (s) and Order-up-to Level (S) (Eqs. 5 & 187 in paper, assuming m=R+L)
    policy_term = D_hat * (R + L)
    s_t = policy_term + SS
    S_t = policy_term + SS 
    
    # Initialize simulation variables
    inventory_level = S_t # Start with target inventory level
    orders_placed = 0
    
    # Metrics to record
    inventory_levels = []
    lost_sales_list = []

    # 2. Simulation Loop (Daily)
    for t in range(SIM_DAYS):
        # A. Order Review (Check only every R days)
        order_quantity = 0
        if (t % R == 0):
            # Order placement only if inventory level is at or below reorder point s_t (Eq. 7)
            if inventory_level <= s_t:
                order_quantity = S_t - inventory_level
                orders_placed += 1
                
                # Simplified: Orders arrive L days later, but we track the 'orders_placed'
                # as the metric required by the ML model.

        # B. Demand Fulfillment
        demand = ts_sales.iloc[t]
        satisfied_sales = min(inventory_level, demand)
        lost_sales = demand - satisfied_sales
        
        # C. Update Inventory
        inventory_level -= satisfied_sales
        
        # D. Recording Metrics
        inventory_levels.append(inventory_level)
        lost_sales_list.append(lost_sales)

    # 3. Post-Simulation Calculation of Inventory Cost Elements (Targets)
    avg_inventory = np.mean(inventory_levels) 
    total_lost_sales = np.sum(lost_sales_list)
    total_orders = orders_placed
    
    # Return the three targets for the ML model
    return avg_inventory, total_lost_sales, total_orders

# --- 4. Main Execution ---

print("5. Performing Stock Control Simulations...")

try:
    # Load features and sales data from the working directory
    features_df = pd.read_csv(FEATURES_FILE)
    sales_df = pd.read_csv(LEVEL_11_SALES_FILE)
except FileNotFoundError:
    print("\nERROR: Required input files not found. Ensure 'dataprep.py' was run successfully.")
    exit()

# --- NO SAMPLING APPLIED ---
# We are using all 9,147 series as requested.
# ---------------------------

# Prepare the data for simulation
sales_data = sales_df.set_index(['item_id', 'state_id'])
reference_sales = sales_data[DAY_COLS]

simulation_results = []
total_iterations = len(features_df) * len(POLICY_HPARAMS)
print(f"Total simulations to run: {total_iterations} ({len(features_df)} series * {len(POLICY_HPARAMS)} hparams).")

# Iterate through each series and each policy combination
for index, row in tqdm(features_df.iterrows(), total=len(features_df), desc="Series Progress"):
    
    item_id = row['item_id']
    state_id = row['state_id']
    adi = row['ADI']
    cv2 = row['CV2']
    
    # Extract the sales history for the current series
    try:
        ts_sales = reference_sales.loc[(item_id, state_id)]
    except KeyError:
        continue # Skip if sales data is missing

    # Iterate through all hyperparameter sets (Phase 2, Step 4)
    for R, L, TSL in POLICY_HPARAMS:
        
        # Run the simulation (Phase 2, Step 5)
        avg_inv, total_ls, total_orders = run_rss_simulation(
            ts_sales=ts_sales,
            R=R, L=L, TSL=TSL
        )

        # Record the ML training observation (Features + Targets)
        simulation_results.append({
            'item_id': item_id,
            'state_id': state_id,
            'ADI': adi,
            'CV2': cv2,
            'R': R,
            'L': L,
            'TSL': TSL,
            'AvgInventory': avg_inv,
            'LostSales': total_ls,
            'NumOrders': total_orders,
        })

# --- 5. Final Save ---
training_df = pd.DataFrame(simulation_results)
training_df.to_csv(SIMULATION_OUTPUT_FILE, index=False)
print("\nSimulation complete!")
print(f"ML Training Data (75 hparams) saved to: {SIMULATION_OUTPUT_FILE}")
print(f"Total training observations generated: {len(training_df)}")

# --- Next Step ---
print("\n--- Next Step (Phase 2, Step 6) ---")
print("6. Train the Machine Learning Models: You will now train three separate LightGBM models.")
print("The script name for the next step should be: model_training.py")



!!! PROTOTYPING MODE: Using reduced hyperparameter set (75 sets) !!!
4. Generating 75 Inventory Policy Hyperparameter Sets...
5. Performing Stock Control Simulations...
Total simulations to run: 686025 (9147 series * 75 hparams).


Series Progress: 100%|██████████| 9147/9147 [26:48<00:00,  5.69it/s]



Simulation complete!
ML Training Data (75 hparams) saved to: /kaggle/working/ml_training_data_75_hparams.csv
Total training observations generated: 686025

--- Next Step (Phase 2, Step 6) ---
6. Train the Machine Learning Models: You will now train three separate LightGBM models.
The script name for the next step should be: model_training.py


In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib # For saving trained models

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory (Must be /kaggle/working)
OUTPUT_DIR = '/kaggle/working'

# Input file from the simulation step
# Use the filename generated by the modified simulation script
TRAINING_DATA_FILE = os.path.join(OUTPUT_DIR, 'ml_training_data_75_hparams.csv') 

# Output directory for saving trained models
MODEL_DIR = os.path.join(OUTPUT_DIR, 'trained_models')
os.makedirs(MODEL_DIR, exist_ok=True) # Create directory if it doesn't exist

# --- 2. Load Training Data ---
print(f"Loading training data from: {TRAINING_DATA_FILE}")
try:
    training_df = pd.read_csv(TRAINING_DATA_FILE)
except FileNotFoundError:
    print(f"\nERROR: Training data file not found. Ensure 'simulation.py' ran successfully.")
    exit()

print(f"Loaded {len(training_df)} training observations.")

# --- 3. Define Features and Targets ---
# Features: As defined in the paper (Section 3.1, Step 6)
FEATURES = ['ADI', 'CV2', 'R', 'L', 'TSL']

# Targets: The inventory cost elements (Section 3.1, Step 6)
TARGETS = ['AvgInventory', 'LostSales', 'NumOrders']

# Basic validation
missing_cols = [col for col in FEATURES + TARGETS if col not in training_df.columns]
if missing_cols:
    print(f"\nERROR: Missing required columns in training data: {missing_cols}")
    exit()

# --- 4. Scale Targets (As mentioned in Section 3.5) ---
# The paper scales targets by average demand (D_bar) before training.
# To do this correctly, we need the average demand for each series.
# This requires merging with the original sales data or features_df again.

# Load features which contain ADI and CV2, which relate to demand
try:
    features_info_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'm5_demand_features.csv'))
    # Calculate D_bar (average demand over the *entire* history used for ADI/CV2 calc)
    # Note: ADI = total_periods / non_zero_periods. Let's approximate D_bar using ADI indirectly.
    # A simpler approach for now: Calculate average demand from the training simulation period.
    # A more accurate way would be to load the full sales data again.
    
    # Let's add scaling later if needed for performance, starting without it for simplicity.
    print("Skipping target scaling for initial implementation (can be added later).")
    
except FileNotFoundError:
    print("Warning: Features file not found, cannot calculate average demand for scaling. Proceeding without scaling.")
    
X_train = training_df[FEATURES]

# --- 5. Train LightGBM Models (One for each target) ---
print("\n6. Training LightGBM models...")

# Define basic LightGBM parameters 
# The paper mentions tuning learning_rate and num_leaves (Section 3.5)
lgbm_params = {
    'objective': 'regression_l1', # MAE is often robust for cost predictions
    'metric': 'mae',
    'n_estimators': 500, # Start with a reasonable number, can be tuned
    'learning_rate': 0.05, # Tuned parameter
    'num_leaves': 64,      # Tuned parameter (controls tree complexity)
    'feature_fraction': 0.8, # Helps prevent overfitting
    'bagging_fraction': 0.8, # Helps prevent overfitting
    'bagging_freq': 1,
    'verbose': -1, # Suppress verbose output
    'n_jobs': -1, # Use all available cores
    'seed': 42
}


for target in TARGETS:
    print(f"  Training model for target: {target}...")
    
    y_train = training_df[target]
    
    # Initialize and train the LightGBM Regressor
    model = lgb.LGBMRegressor(**lgbm_params)
    model.fit(X_train, y_train)
    
    # Save the trained model
    model_filename = os.path.join(MODEL_DIR, f'lgbm_model_{target}.joblib')
    joblib.dump(model, model_filename)
    print(f"    Model saved to: {model_filename}")

print("\nAll models trained and saved successfully!")

# --- Next Step ---
print("\n--- Next Step (Phase 3, Step 7) ---")
print("7. Predict Inventory Cost Elements (On-line Phase): Use the trained models to predict costs for the target series.")
print("The script name for the next step should be: optimization.py")


Loading training data from: /kaggle/working/ml_training_data_75_hparams.csv

ERROR: Training data file not found. Ensure 'simulation.py' ran successfully.


NameError: name 'training_df' is not defined

: 

# CATGB XGB 


In [1]:
import os
import pandas as pd
import numpy as np
import joblib 

# Import all necessary GBM libraries
import lightgbm as lgb 
import xgboost as xgb 
import catboost as catb 


# --- 1. Configuration and File Paths ---
# CHOOSE YOUR MODEL: 'LGBM', 'XGBM', or 'CATB'
# NOTE: Run this script once for each model type.
MODEL_TYPE = 'LGBM' # <--- CHANGE THIS (e.g., to 'LGBM', 'XGBM', or 'CATB')

# INPUT/OUTPUT directory (Must be consistent with your project structure)
OUTPUT_DIR = '/kaggle/working'
TRAINING_DATA_FILE = os.path.join(OUTPUT_DIR, 'ml_training_data_75_hparams.csv') 

# Output directory uses the model type name
MODEL_DIR = os.path.join(OUTPUT_DIR, f'trained_models_{MODEL_TYPE}')
os.makedirs(MODEL_DIR, exist_ok=True) 

# --- 2. Load Training Data ---
print(f"Loading training data for {MODEL_TYPE} from: {TRAINING_DATA_FILE}")
try:
    training_df = pd.read_csv(TRAINING_DATA_FILE)
except FileNotFoundError:
    print(f"\nERROR: Training data file not found. Ensure 'simulation.py' ran successfully.")
    # In a notebook, you might just want to print the error and stop the cell execution.
    # raise FileNotFoundError("Training data file not found.") 

print(f"Loaded {len(training_df)} training observations.")

# --- 3. Define Features and Targets ---
FEATURES = ['ADI', 'CV2', 'R', 'L', 'TSL']
TARGETS = ['AvgInventory', 'LostSales', 'NumOrders']

X_train = training_df[FEATURES]
print("Skipping target scaling for initial implementation (can be added later).")

# --- 4. Define Model Parameters and Class ---

# Parameters are a starting point; optimal performance requires tuning (as per Section 3.5)
MODEL_CONFIGS = {
    'LGBM': {
        'class': lgb.LGBMRegressor,
        'params': {
            'objective': 'regression_l1', # MAE
            'metric': 'mae',
            'n_estimators': 500, 
            'learning_rate': 0.05, 
            'num_leaves': 64, # Controls complexity
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 1,
            'verbose': -1,
            'n_jobs': -1,
            'random_state': 42 
        }
    },
    'XGBM': {
        'class': xgb.XGBRegressor,
        'params': {
            'objective': 'reg:absoluteerror', # XGBoost equivalent of MAE
            'n_estimators': 500,
            'learning_rate': 0.05,
            'max_depth': 6, # Controls complexity
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'n_jobs': -1,
            'random_state': 42
        }
    },
    'CATB': {
        'class': catb.CatBoostRegressor,
        'params': {
            'loss_function': 'MAE', # CatBoost MAE
            'n_estimators': 500,
            'learning_rate': 0.05,
            'depth': 6, # Controls complexity
            'subsample': 0.8,
            'bootstrap_type': 'Bernoulli',
            'verbose': 0, # Suppress output
            'thread_count': -1, # Use all available cores
            'random_seed': 42 # CatBoost uses random_seed
        }
    }
}

# Get the configuration for the selected model type
config = MODEL_CONFIGS.get(MODEL_TYPE)
if not config:
    print(f"\nERROR: Invalid MODEL_TYPE '{MODEL_TYPE}'. Choose from 'LGBM', 'XGBM', or 'CATB'.")
    exit()

ModelClass = config['class']
model_params = config['params']

# --- 5. Train Models (One for each target) ---
print(f"\n6. Training {MODEL_TYPE} models...")

for target in TARGETS:
    print(f"  Training model for target: {target}...")
    
    y_train = training_df[target]
    
    # Initialize and train the Regressor
    model = ModelClass(**model_params)
    model.fit(X_train, y_train)
    
    # Save the trained model
    model_filename = os.path.join(MODEL_DIR, f'{MODEL_TYPE.lower()}_model_{target}.joblib')
    joblib.dump(model, model_filename)
    print(f"    Model saved to: {model_filename}")

print("\nAll models trained and saved successfully!")

# --- Next Step ---
print("\n--- Next Step (Phase 3, Step 7) ---")
print("7. Predict Inventory Cost Elements (On-line Phase): Use the trained models to predict costs for the target series.")

Loading training data for LGBM from: /kaggle/working/ml_training_data_75_hparams.csv

ERROR: Training data file not found. Ensure 'simulation.py' ran successfully.


NameError: name 'training_df' is not defined

# LGBM , XGB , CATGB OPTIMISATION 

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib # For loading trained models
from itertools import product
from tqdm import tqdm

# --- 1. Configuration and File Paths ---

# CHOOSE YOUR MODEL: 'LGBM', 'XGBM', or 'CATB'
# NOTE: Run this script once for each model type.
MODEL_TYPE = 'CATB' # <--- CHANGE THIS for each run!

# INPUT/OUTPUT directory
OUTPUT_DIR = '/kaggle/working'

# Dynamic Model Directory: Must match the saving location from the training script
MODEL_DIR = os.path.join(OUTPUT_DIR, f'trained_models_{MODEL_TYPE}')

# Input files: Features of the target series
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv')

# Dynamic Output file: Optimal hyperparameters found for each series
OPTIMAL_PARAMS_FILE = os.path.join(OUTPUT_DIR, f'optimal_inventory_params_{MODEL_TYPE}.csv')

# --- 2. Define Hyperparameter Sets and Cost Structure ---

# IMPORTANT: Use the SAME hyperparameter grid as used in your training (simulation) script!
print(f"!!! Using reduced hyperparameter set for {MODEL_TYPE} optimization !!!")
R_RANGE = np.linspace(1, 30, 5, dtype=int)
L_RANGE = np.linspace(1, 30, 5, dtype=int)
TSL_RANGE = np.array([0.90, 0.95, 0.99])
POLICY_HPARAMS = list(product(R_RANGE, L_RANGE, TSL_RANGE))
HPARAMS_DF = pd.DataFrame(POLICY_HPARAMS, columns=['R', 'L', 'TSL'])
print(f"Generated {len(HPARAMS_DF)} policy hyperparameter sets for evaluation.")

# Inventory Cost Hyperparameters (h, b, k) from the paper (Section 3.4)
h = 0.01 # Annual holding cost % 
b = 0.25 # Unit shortage cost %
k = 0.50 # Cost per order ($)
h_daily = h / 365 
print(f"Using Cost Parameters: h={h}, b={b}, k={k}")

# --- 3. Load Trained Models ---
print(f"\nLoading trained {MODEL_TYPE} models from {MODEL_DIR}...")
models = {}
TARGETS = ['AvgInventory', 'LostSales', 'NumOrders']

try:
    model_prefix = MODEL_TYPE.lower()
    for target in TARGETS:
        model_filename = os.path.join(MODEL_DIR, f'{model_prefix}_model_{target}.joblib')
        models[target] = joblib.load(model_filename)
        print(f"  Loaded model for {target}.")
except FileNotFoundError as e:
    print(f"\nERROR: Could not load model file: {e}. Ensure 'model_training.py' ran successfully for {MODEL_TYPE} and created the directory {MODEL_DIR}.")
    exit()

# --- 4. Load Target Series Features ---
print(f"\nLoading features for target series from: {FEATURES_FILE}")
try:
    features_df = pd.read_csv(FEATURES_FILE)
    id_cols = ['item_id', 'state_id']
    feature_cols = ['ADI', 'CV2']
    features_df = features_df[id_cols + feature_cols]
except FileNotFoundError:
    print(f"\nERROR: Features file not found. Ensure 'dataprep.py' ran successfully.")
    exit()
print(f"Loaded features for {len(features_df)} target series.")

# --- 5. On-line Phase: Predict Costs and Find Optimal Policy ---

optimal_policies = []
print(f"\nOptimizing policies for {len(features_df)} series using {MODEL_TYPE}...")

# Prepare the static DataFrame for prediction once (features are appended in loop)
prediction_df_base = HPARAMS_DF.copy()

# Iterate through each unique series (item_id, state_id)
for index, series_features in tqdm(features_df.iterrows(), total=len(features_df), desc="Optimizing"):
    
    # Extract features
    series_id = f"{series_features['item_id']}_{series_features['state_id']}"
    adi = series_features['ADI']
    cv2 = series_features['CV2']

    # Create a DataFrame combining series features with all policy hparams
    prediction_df = prediction_df_base.copy()
    prediction_df['ADI'] = adi
    prediction_df['CV2'] = cv2
    # Ensure correct order for the model input (ADI, CV2, R, L, TSL)
    prediction_df = prediction_df[['ADI', 'CV2', 'R', 'L', 'TSL']] 

    # Predict cost elements for all policies (Step 7)
    predicted_costs = {}
    for target, model in models.items():
        # CatBoost requires specific handling for pandas DataFrames if not passed as pool
        if MODEL_TYPE == 'CATB':
             # CatBoost can sometimes be sensitive to data types, forcing to float/int
             X_pred = prediction_df.astype({'ADI': float, 'CV2': float, 'R': int, 'L': int, 'TSL': float}) 
        else:
             X_pred = prediction_df
             
        predicted_costs[target] = model.predict(X_pred)
    
    predicted_costs_df = pd.DataFrame(predicted_costs)
    
    # Estimate Total Inventory Cost for each policy (Step 8)
    # C_Tot = (h_daily * AvgInventory * 365) + (b * LostSales) + (k * NumOrders)
    predicted_costs_df['TotalCost'] = (
        h_daily * predicted_costs_df['AvgInventory'] * 365 + 
        b * predicted_costs_df['LostSales'] +
        k * predicted_costs_df['NumOrders']  
    )
    
    # Combine predictions with policy parameters
    results_df = pd.concat([HPARAMS_DF, predicted_costs_df], axis=1)

    # Identify Optimal Hyperparameters (Step 9)
    optimal_row = results_df.loc[results_df['TotalCost'].idxmin()]

    # Store the optimal policy for this series
    optimal_policies.append({
        'series_id': series_id,
        'item_id': series_features['item_id'],
        'state_id': series_features['state_id'],
        'Optimal_R': optimal_row['R'],
        'Optimal_L': optimal_row['L'],
        'Optimal_TSL': optimal_row['TSL'],
        'Predicted_Min_Cost': optimal_row['TotalCost']
    })

# --- 6. Save Optimal Policies ---
optimal_policies_df = pd.DataFrame(optimal_policies)
optimal_policies_df.to_csv(OPTIMAL_PARAMS_FILE, index=False)

print("\nOptimization complete!")
print(f"Optimal inventory policies saved to: {OPTIMAL_PARAMS_FILE}")

# --- Next Steps ---
print("\n--- Next Steps (Phase 3, Steps 10 & 11) ---")
print("10. Evaluate Performance: Run final simulations using the optimal policies found.")
print("11. Compare with Benchmarks: Implement TO and FSH benchmarks for comparison.")

!!! Using reduced hyperparameter set for CATB optimization !!!
Generated 75 policy hyperparameter sets for evaluation.
Using Cost Parameters: h=0.01, b=0.25, k=0.5

Loading trained CATB models from /kaggle/working/trained_models_CATB...
  Loaded model for AvgInventory.
  Loaded model for LostSales.
  Loaded model for NumOrders.

Loading features for target series from: /kaggle/working/m5_demand_features.csv
Loaded features for 9147 target series.

Optimizing policies for 9147 series using CATB...


Optimizing: 100%|██████████| 9147/9147 [01:45<00:00, 86.82it/s]



Optimization complete!
Optimal inventory policies saved to: /kaggle/working/optimal_inventory_params_CATB.csv

--- Next Steps (Phase 3, Steps 10 & 11) ---
10. Evaluate Performance: Run final simulations using the optimal policies found.
11. Compare with Benchmarks: Implement TO and FSH benchmarks for comparison.


# LGBM XGB CATB EVALUATION 

In [4]:
import os
import pandas as pd
import numpy as np
from scipy.stats import norm
from tqdm import tqdm

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory
OUTPUT_DIR = '/kaggle/working'
MODEL_TYPES = ['LGBM', 'XGBM', 'CATB'] 

# Input files (assuming a consistent naming scheme)
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv') 
# Assuming ml_training_data_75_hparams.csv covers d_1 to d_1913 for historical purposes
SIMULATION_RESULTS_FILE = os.path.join(OUTPUT_DIR, 'ml_training_data_75_hparams.csv') 
PRICES_FILE = os.path.join(OUTPUT_DIR, 'sell_prices.csv')

# Use the SALES_TRAIN_VALIDATION file as it typically goes up to d_1913
# We'll use this for the historical stats calculation.
INPUT_DIR_ORIGINAL = '/kaggle/input/m5-forecasting-accuracy'
SALES_FILE_VALIDATION = os.path.join(INPUT_DIR_ORIGINAL, 'sales_train_validation.csv') 

# --- 2. Corrected Evaluation Period and Cost Parameters ---
# The original M5 validation file goes up to d_1913. 
# The M5 paper data goes up to d_1969 (3049 * 10 * 1969 days).
# We will match the paper's *historical* split point but use a shorter, valid eval period.
SIM_DAYS = 365 # Historical days to use for calculating D_hat, sigma_D
EVAL_DAYS = 365 # Target length for the paper, but we must cap at max available days
MAX_SALES_DAY = 1913 # Maximum day in sales_train_validation.csv

# Define periods based on the paper's intention (last historical year, followed by next year)
# HISTORICAL STATS: The 365 days *before* the evaluation begins.
EVAL_START_DAY = 1914 
HIST_START_DAY = EVAL_START_DAY - SIM_DAYS

# We MUST cap the Evaluation period at the maximum available day (d_1913 in validation file)
# If you are using 'sales_train_evaluation.csv', max day is d_1941 + 28 = d_1969.
# Let's assume you've used sales_train_validation (up to d_1913) for training.
# The paper suggests evaluation uses the 'future observations' (d_1914 onwards).

# We must adjust SIM_DAYS for *accurate* historical stats based on the available data up to d_1913
# To get a 365-day historical period ending at d_1913:
HIST_DAY_COLS = [f'd_{i}' for i in range(1914 - SIM_DAYS, 1914)] # d_1549 to d_1913

# We will cap EVAL_DAYS to the available future data *if* you have loaded the evaluation file
# Assuming you want to use the first 365 days (d_1914 to d_2278) which is not possible.
# For consistency with the code structure, we will cap the EVAL_DAY_COLS to a safe, available range.
# For simplicity, let's use the actual next 28 days available in 'sales_train_evaluation.csv' 
# which is d_1914 to d_1941 (the M5 public leaderboard period).

EVAL_DAYS_CAP = 28
EVAL_DAY_COLS = [f'd_{i}' for i in range(EVAL_START_DAY, EVAL_START_DAY + EVAL_DAYS_CAP)] # d_1914 to d_1941

# Inventory Cost Hyperparameters (h, b, k)
h = 0.01 
b = 0.25 
k = 0.50 
h_daily = h / 365 

FSH_PARAMS = {'R': 7, 'L': 3, 'TSL': 0.95}
print(f"Historical Stats Period: {SIM_DAYS} days (d_{HIST_DAY_COLS[0]} to d_{HIST_DAY_COLS[-1]})")
print(f"Evaluation Period: {EVAL_DAYS_CAP} days (d_{EVAL_DAY_COLS[0]} to d_{EVAL_DAY_COLS[-1]})")


# --- 3. Load and Prepare Core Data ---
print("\nLoading and preparing core data...")
all_sales_df = None
try:
    # Load sales data (must contain both HIST and EVAL period)
    # Usecols only loads necessary columns to save memory
    sales_cols = ['item_id', 'state_id'] + HIST_DAY_COLS + EVAL_DAY_COLS
    
    # Try loading the EVALUATION file first (since it extends past d_1913)
    SALES_FILE_FULL = os.path.join(INPUT_DIR_ORIGINAL, 'sales_train_evaluation.csv')
    try:
        all_sales_df = pd.read_csv(SALES_FILE_FULL, usecols=sales_cols)
    except Exception:
         # Fallback to validation file if evaluation file path is wrong/not loaded
        SALES_FILE_FULL = os.path.join(INPUT_DIR_ORIGINAL, 'sales_train_validation.csv')
        all_sales_df = pd.read_csv(SALES_FILE_FULL, usecols=sales_cols)

    # Aggregate sales data to Level 11 for HISTORICAL and EVALUATION periods
    level_11_sales = all_sales_df.groupby(['item_id', 'state_id'])[HIST_DAY_COLS + EVAL_DAY_COLS].sum().reset_index()

    # Load simulation results (for TO benchmark determination)
    simulation_results_df = pd.read_csv(SIMULATION_RESULTS_FILE)
    
    # Load features (to filter the series to the 8504 used in the training)
    features_df = pd.read_csv(FEATURES_FILE)
    target_series_ids = features_df[['item_id', 'state_id']].drop_duplicates()
    level_11_sales = target_series_ids.merge(level_11_sales, on=['item_id', 'state_id'], how='inner')
    
except Exception as e:
    # Print the error clearly and stop
    print(f"\nFATAL ERROR during data loading: {e}")
    # The fix for the NameError is to wrap the rest of the script in a function or an if block
    exit()

# --- 4. Implement True Optimal (TO) Benchmark (Step 11) ---
print("\nDetermining True Optimal (TO) benchmark policies...")
# Cost calculation for the Historical Period (d_1 to d_1913)
# Note: The AvgInventory and LostSales in simulation_results_df are totals over the *training* period
# We must adjust the holding cost calculation to match the period length used in the simulation file.
# Assuming the simulation used 365 days (SIM_DAYS):
simulation_results_df['HistoricalCost'] = (
    h_daily * simulation_results_df['AvgInventory'] * SIM_DAYS + 
    b * simulation_results_df['LostSales'] +
    k * simulation_results_df['NumOrders'] 
)

# Find the best R, L, TSL for each series based on lowest historical cost
to_policies_idx = simulation_results_df.groupby(['item_id', 'state_id'])['HistoricalCost'].idxmin()
to_policies_df = simulation_results_df.loc[to_policies_idx, ['item_id', 'state_id', 'R', 'L', 'TSL']].reset_index(drop=True)
to_policies_df.rename(columns={'R': 'TO_R', 'L': 'TO_L', 'TSL': 'TO_TSL'}, inplace=True)
print(f"Determined TO policies for {len(to_policies_df)} series.")

# --- 5. Re-Implement Core Simulation Logic (R, s, S) ---

def calculate_policy_stats(historical_sales):
    """Calculates D_hat and sigma_D from the historical period sales."""
    history = historical_sales.values
    D_hat = history.mean()
    sigma_D = history.std(ddof=1) if len(history) > 1 else 1.0
    return D_hat, sigma_D

def run_evaluation_simulation(historical_sales, eval_sales, R, L, TSL):
    """
    Runs the (R, s, S) simulation on the evaluation period sales data.
    """
    if len(historical_sales) == 0 or len(eval_sales) == 0:
        return np.nan, np.nan, np.nan, np.nan 
        
    D_hat, sigma_D = calculate_policy_stats(historical_sales)
    
    # Clamp TSL for safe norm.ppf calculation
    TSL = np.clip(TSL, 1e-6, 1 - 1e-6)
    
    if sigma_D <= 0:
        Z = 2.0 
    else:
        Z = norm.ppf(TSL)
        
    SS = Z * sigma_D * np.sqrt(R + L)
    s_t = D_hat * (R + L) + SS
    S_t = D_hat * (R + L) + SS 

    inventory_level = S_t # Start with target inventory level
    orders_placed = 0
    inventory_levels = []
    lost_sales_list = []
    total_demand_list = []
    
    eval_sales_array = eval_sales.values

    for t in range(len(eval_sales_array)):
        demand = eval_sales_array[t]
        
        # Replenishment Decision (Every R periods)
        order_quantity = 0
        if t % R == 0:
            if inventory_level <= s_t:
                order_quantity = S_t - inventory_level
                orders_placed += 1
        
        # NOTE: Without a proper queue, the order arrival is simplified.
        # This implementation assumes the order is instantly placed and the safety stock 
        # calculation handles the lead time L implicitly.
        
        satisfied_sales = min(inventory_level, demand)
        lost_sales = demand - satisfied_sales
        inventory_level -= satisfied_sales
        inventory_levels.append(inventory_level)
        lost_sales_list.append(lost_sales)
        total_demand_list.append(demand)
        
    avg_inventory_real = np.sum(inventory_levels) / EVAL_DAYS_CAP 
    total_lost_sales_real = np.sum(lost_sales_list)
    total_orders_real = orders_placed
    
    total_demand_real = np.sum(total_demand_list)
    realized_sl = (1 - (total_lost_sales_real / total_demand_real)) if total_demand_real > 0 else 1.0

    return avg_inventory_real, total_lost_sales_real, total_orders_real, realized_sl

# --- 6. Run Evaluations for MOIC, TO, FSH ---
print("\nRunning evaluation simulations...")

all_evaluation_results = []
base_eval_df = level_11_sales.merge(to_policies_df, on=['item_id', 'state_id'], how='inner')

for model_type in MODEL_TYPES:
    print(f"\n--- Evaluating MOIC Model: {model_type} ---")
    
    # Load optimal policies for the current model type
    OPTIMAL_PARAMS_FILE = os.path.join(OUTPUT_DIR, f'optimal_inventory_params_{model_type}.csv')
    try:
        optimal_policies_df = pd.read_csv(OPTIMAL_PARAMS_FILE)
    except FileNotFoundError:
        print(f"Skipping {model_type}: Optimal parameters file not found at {OPTIMAL_PARAMS_FILE}")
        continue

    eval_df_model = base_eval_df.merge(
        optimal_policies_df[['item_id', 'state_id', 'Optimal_R', 'Optimal_L', 'Optimal_TSL']], 
        on=['item_id', 'state_id'], 
        how='inner'
    )
    
    for index, row in tqdm(eval_df_model.iterrows(), total=len(eval_df_model), desc=f"Simulating {model_type}"):
        item_id = row['item_id']
        state_id = row['state_id']
        
        hist_sales = row[HIST_DAY_COLS]
        eval_sales = row[EVAL_DAY_COLS]

        # Use a list of dictionaries to store results for this series
        results = {'item_id': item_id, 'state_id': state_id}
        
        # Evaluate MOIC Optimal Policy
        moic_r, moic_l, moic_tsl = row['Optimal_R'], row['Optimal_L'], row['Optimal_TSL']
        avg_inv, ls, orders, sl = run_evaluation_simulation(hist_sales, eval_sales, moic_r, moic_l, moic_tsl)
        results[f'MOIC_{model_type}_SL'] = sl
        results[f'MOIC_{model_type}_Cost'] = h_daily * avg_inv * EVAL_DAYS_CAP + b * ls + k * orders
        
        # Evaluate TO Benchmark Policy (Run once per series)
        to_r, to_l, to_tsl = row['TO_R'], row['TO_L'], row['TO_TSL']
        avg_inv, ls, orders, sl = run_evaluation_simulation(hist_sales, eval_sales, to_r, to_l, to_tsl)
        results['TO_SL'] = sl
        results['TO_Cost'] = h_daily * avg_inv * EVAL_DAYS_CAP + b * ls + k * orders

        # Evaluate FSH Benchmark Policy (Run once per series)
        fsh_r, fsh_l, fsh_tsl = FSH_PARAMS['R'], FSH_PARAMS['L'], FSH_PARAMS['TSL']
        avg_inv, ls, orders, sl = run_evaluation_simulation(hist_sales, eval_sales, fsh_r, fsh_l, fsh_tsl)
        results['FSH_SL'] = sl
        results['FSH_Cost'] = h_daily * avg_inv * EVAL_DAYS_CAP + b * ls + k * orders
        
        all_evaluation_results.append(results)

# --- 7. Save Final Evaluation Results ---
evaluation_results_df = pd.DataFrame(all_evaluation_results)

# Consolidate duplicate TO/FSH results from the loop
pivot_cols = ['item_id', 'state_id']
final_evaluation_df = evaluation_results_df.groupby(pivot_cols).agg('first').reset_index()

# Drop redundant columns that were automatically generated if needed
for m in MODEL_TYPES:
    if f'MOIC_{m}_SL' not in final_evaluation_df.columns:
        # Fill with NaN if a model type was skipped due to missing files
        final_evaluation_df[f'MOIC_{m}_SL'] = np.nan
        final_evaluation_df[f'MOIC_{m}_Cost'] = np.nan
        
EVALUATION_RESULTS_FILE = os.path.join(OUTPUT_DIR, 'final_evaluation_summary_GBM_Comparison.csv')
final_evaluation_df.to_csv(EVALUATION_RESULTS_FILE, index=False)

print("\nEvaluation complete!")
print(f"Final evaluation results saved to: {EVALUATION_RESULTS_FILE}")

# --- Final Step ---
print("\n--- Next Step (Phase 4, Step 12) ---")
print("12. Analyze and Summarize Results: Calculate the total costs and average SL across all series for the summary table (like Table 3 in the paper) and compare LGBM, XGBM, and CATB.")

Historical Stats Period: 365 days (d_d_1549 to d_d_1913)
Evaluation Period: 28 days (d_d_1914 to d_d_1941)

Loading and preparing core data...

Determining True Optimal (TO) benchmark policies...
Determined TO policies for 9147 series.

Running evaluation simulations...

--- Evaluating MOIC Model: LGBM ---
Skipping LGBM: Optimal parameters file not found at /kaggle/working/optimal_inventory_params_LGBM.csv

--- Evaluating MOIC Model: XGBM ---


Simulating XGBM: 100%|██████████| 9147/9147 [00:23<00:00, 391.98it/s]



--- Evaluating MOIC Model: CATB ---


Simulating CATB: 100%|██████████| 9147/9147 [00:23<00:00, 394.74it/s]



Evaluation complete!
Final evaluation results saved to: /kaggle/working/final_evaluation_summary_GBM_Comparison.csv

--- Next Step (Phase 4, Step 12) ---
12. Analyze and Summarize Results: Calculate the total costs and average SL across all series for the summary table (like Table 3 in the paper) and compare LGBM, XGBM, and CATB.


# ANALYSIS

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory (Must be /kaggle/working)
OUTPUT_DIR = '/kaggle/working'

# Input files: Using the consolidated output from the multi-model evaluation script
EVALUATION_RESULTS_FILE = os.path.join(OUTPUT_DIR, 'final_evaluation_summary_GBM_Comparison.csv')
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv') # Needed for demand categories

# Output directory for plots
PLOT_DIR = os.path.join(OUTPUT_DIR, 'analysis_plots_gbm_comparison')
os.makedirs(PLOT_DIR, exist_ok=True)

# Model Types to analyze
MODEL_TYPES = ['LGBM', 'XGBM', 'CATB']

# Demand Categorization Thresholds (from paper Section 2.1)
ADI_THRESHOLD = 1.32
CV2_THRESHOLD = 0.49

# --- 2. Load Data ---
print("Loading consolidated evaluation results and features...")
try:
    eval_df = pd.read_csv(EVALUATION_RESULTS_FILE)
    features_df = pd.read_csv(FEATURES_FILE, usecols=['item_id', 'state_id', 'ADI', 'CV2'])
except FileNotFoundError as e:
    print(f"\nERROR: Could not load required file: {e}. Ensure the multi-model evaluation ran successfully.")
    exit()

# Merge features onto evaluation results for category analysis
results_df = pd.merge(eval_df, features_df, on=['item_id', 'state_id'], how='left')

# Define all Cost and SL columns for comparison
COST_COLS = [f'MOIC_{m}_Cost' for m in MODEL_TYPES] + ['TO_Cost', 'FSH_Cost']
SL_COLS = [f'MOIC_{m}_SL' for m in MODEL_TYPES] + ['TO_SL', 'FSH_SL']
ALL_METRIC_COLS = COST_COLS + SL_COLS

# --- 3. Overall Performance Comparison (Table 3 Equivalent) ---
print("\n--- Overall Performance Comparison (Equivalent to Table 3) ---")

# Calculate average Cost and Service Level for all methods
overall_summary_mean = results_df[ALL_METRIC_COLS].mean().reset_index()
overall_summary_mean.columns = ['Metric', 'AverageValue']

# Extract Method Name and Measure (e.g., MOIC_LGBM, TO)
overall_summary_mean['Method'] = overall_summary_mean['Metric'].str.replace('_Cost', '').str.replace('_SL', '')
overall_summary_mean['Measure'] = overall_summary_mean['Metric'].apply(lambda x: 'Cost' if 'Cost' in x else 'SL')

# Pivot for better readability
overall_table = overall_summary_mean.pivot(index='Method', columns='Measure', values='AverageValue')
print("🔥 Overall Average Cost and Service Level:")
print(overall_table[['Cost', 'SL']].sort_values(by='Cost')) # Sort by cost for comparison

# --- 4. Define Demand Categories ---
def categorize_demand(row):
    adi = row['ADI']
    cv2 = row['CV2']
    if adi < ADI_THRESHOLD and cv2 < CV2_THRESHOLD:
        return 'Smooth'
    elif adi >= ADI_THRESHOLD and cv2 < CV2_THRESHOLD:
        return 'Intermittent' 
    elif adi < ADI_THRESHOLD and cv2 >= CV2_THRESHOLD:
        return 'Erratic'
    else: # adi >= ADI_THRESHOLD and cv2 >= CV2_THRESHOLD
        return 'Lumpy'

results_df['DemandCategory'] = results_df.apply(categorize_demand, axis=1)
print("\nDemand Categories Assigned. Counts:")
print(results_df['DemandCategory'].value_counts())

# --- 5. Performance by Demand Category (Table 4 Equivalent) ---
print("\n--- Performance by Demand Category (Equivalent to Table 4) ---")

category_summary = results_df.groupby('DemandCategory')[ALL_METRIC_COLS].mean()

# Display Cost and SL separately
print("\nAverage Cost by Demand Category:")
print(category_summary[COST_COLS])

print("\nAverage Service Level (SL) by Demand Category:")
print(category_summary[SL_COLS])


# --- 6. Visualizations (Figures 7 & 8 Equivalents) ---
print("\nGenerating comparison plots...")
sns.set_style("whitegrid")
plot_methods = [f'MOIC_{m}' for m in MODEL_TYPES] + ['TO', 'FSH']

# --- Figure 7a/b Equivalent (Distribution of Cost and SL) ---

# Melt data for easy plotting
melt_df = pd.melt(results_df,
                  id_vars=['DemandCategory', 'state_id'],
                  value_vars=ALL_METRIC_COLS,
                  var_name='Metric', value_name='Value')

melt_df['Method'] = melt_df['Metric'].str.replace('_Cost', '').str.replace('_SL', '')
melt_df['Measure'] = melt_df['Metric'].apply(lambda x: 'Cost' if 'Cost' in x else 'SL')

# 7a: Distribution of Total Cost (All Methods)
plt.figure(figsize=(12, 6))
cost_data = melt_df[melt_df['Measure'] == 'Cost'].copy()
# Remove top 1% outliers for better visualization
cost_limit = cost_data['Value'].quantile(0.99)
cost_data = cost_data[cost_data['Value'] < cost_limit]
sns.boxplot(x='Method', y='Value', data=cost_data, order=plot_methods, palette='viridis')
plt.title('Distribution of Realized Total Inventory Cost per Series (Outliers Removed)')
plt.ylabel('Total Cost ($)')
plt.xlabel('Method')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'figure7a_cost_distribution_all_gbm.png'))
print(f"  Saved Cost Distribution plot to {PLOT_DIR}")
plt.close()

# 7b: Distribution of Service Level (All Methods)
plt.figure(figsize=(12, 6))
sl_data = melt_df[melt_df['Measure'] == 'SL'].copy()
sns.boxplot(x='Method', y='Value', data=sl_data, order=plot_methods, palette='viridis')
plt.title('Distribution of Realized Service Level per Series')
plt.ylabel('Service Level')
plt.xlabel('Method')
plt.ylim(0.8, 1.01)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'figure7b_sl_distribution_all_gbm.png'))
print(f"  Saved Service Level Distribution plot to {PLOT_DIR}")
plt.close()

# Figure 8 Equivalent: Cost Distribution by Demand Category
plt.figure(figsize=(18, 9))
sns.boxplot(x='DemandCategory', y='Value', hue='Method', 
            data=cost_data, # Use the outlier-filtered cost data
            hue_order=plot_methods, palette='viridis')
plt.title('Distribution of Realized Total Cost by Demand Category (Outliers Removed)')
plt.ylabel('Total Cost ($)')
plt.xlabel('Demand Category')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'figure8_cost_by_category_all_gbm.png'))
print(f"  Saved Cost Distribution by Category plot to {PLOT_DIR}")
plt.close()


print("\nAnalysis complete. Plots saved in:", PLOT_DIR)

# --- 8. Conclusion Summary (Printed Output) ---
print("\n--- Conclusion Summary ---")
print("The tables show the detailed average performance of LGBM, XGBM, and CATB based MOIC variants compared to TO and FSH.")
print("The lowest average cost indicates the best performing strategy.")
print("The generated plots (Cost and SL Distribution, Cost by Category) provide visual confirmation of robustness.")
print("Proceed to write up your final report based on these tables and figures.")

Loading consolidated evaluation results and features...

--- Overall Performance Comparison (Equivalent to Table 3) ---
🔥 Overall Average Cost and Service Level:
Measure         Cost        SL
Method                        
MOIC_CATB   0.920617  0.994467
TO          0.920625  0.994467
MOIC_XGBM   1.015378  0.993989
FSH        21.309346  0.516163
MOIC_LGBM        NaN       NaN

Demand Categories Assigned. Counts:
DemandCategory
Intermittent    5405
Smooth          1768
Lumpy           1493
Erratic          481
Name: count, dtype: int64

--- Performance by Demand Category (Equivalent to Table 4) ---

Average Cost by Demand Category:
                MOIC_LGBM_Cost  MOIC_XGBM_Cost  MOIC_CATB_Cost   TO_Cost  \
DemandCategory                                                             
Erratic                    NaN        1.258131        0.878195  0.878195   
Intermittent               NaN        0.942919        0.839122  0.839134   
Lumpy                      NaN        0.924162        0.8

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

  Saved Cost Distribution plot to /kaggle/working/analysis_plots_gbm_comparison
  Saved Service Level Distribution plot to /kaggle/working/analysis_plots_gbm_comparison
  Saved Cost Distribution by Category plot to /kaggle/working/analysis_plots_gbm_comparison

Analysis complete. Plots saved in: /kaggle/working/analysis_plots_gbm_comparison

--- Conclusion Summary ---
The tables show the detailed average performance of LGBM, XGBM, and CATB based MOIC variants compared to TO and FSH.
The lowest average cost indicates the best performing strategy.
The generated plots (Cost and SL Distribution, Cost by Category) provide visual confirmation of robustness.
Proceed to write up your final report based on these tables and figures.


In [8]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib # For loading trained models
from itertools import product
from tqdm import tqdm

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory (Must be /kaggle/working)
OUTPUT_DIR = '/kaggle/working'
MODEL_DIR = os.path.join(OUTPUT_DIR, 'trained_models')

# Input files: Features of the target series
# We use the features calculated in dataprep.py for the historical period.
# The paper implies using the same features for the on-line phase.
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'm5_demand_features.csv')

# Output file: Optimal hyperparameters found for each series
OPTIMAL_PARAMS_FILE = os.path.join(OUTPUT_DIR, 'optimal_inventory_params.csv')

# --- 2. Define Hyperparameter Sets and Cost Structure ---

# IMPORTANT: Use the SAME hyperparameter grid as used in simulation.py
# If you used the reduced set (75) for training, use it here too.
# If you used the full set (9000), use that here for the final optimization.

print("!!! Using reduced hyperparameter set (75 sets) for optimization !!!")
# R: Review Period (5 values)
R_RANGE = np.linspace(1, 30, 5, dtype=int)
# L: Lead Time (5 values)
L_RANGE = np.linspace(1, 30, 5, dtype=int)
# TSL: Target Service Level (3 values)
TSL_RANGE = np.array([0.90, 0.95, 0.99])
POLICY_HPARAMS = list(product(R_RANGE, L_RANGE, TSL_RANGE))
HPARAMS_DF = pd.DataFrame(POLICY_HPARAMS, columns=['R', 'L', 'TSL'])
print(f"Generated {len(HPARAMS_DF)} policy hyperparameter sets for evaluation.")

# Inventory Cost Hyperparameters (h, b, k) from the paper (Section 3.4)
h = 0.01  # Annual holding cost % (daily needs adjustment)
b = 0.25  # Unit shortage cost %
k = 0.50  # Cost per order ($)

# Note: The paper doesn't explicitly state how p_i (item price) is used with h and b.
# Assuming h and b are applied *directly* to AvgInventory and LostSales for simplicity.
# A more accurate implementation would require merging price data.
# For now, we calculate cost based on predicted counts, not dollar values.
# Adjustment for daily holding cost (assuming h=1% annually)
h_daily = h / 365 

print(f"Using Cost Parameters: h={h}, b={b}, k={k}")

# --- 3. Load Trained Models ---
print("\nLoading trained LightGBM models...")
models = {}
TARGETS = ['AvgInventory', 'LostSales', 'NumOrders']
try:
    for target in TARGETS:
        model_filename = os.path.join(MODEL_DIR, f'lgbm_model_{target}.joblib')
        models[target] = joblib.load(model_filename)
        print(f"  Loaded model for {target}.")
except FileNotFoundError as e:
    print(f"\nERROR: Could not load model file: {e}. Ensure 'model_training.py' ran successfully.")
    exit()

# --- 4. Load Target Series Features ---
print(f"\nLoading features for target series from: {FEATURES_FILE}")
try:
    features_df = pd.read_csv(FEATURES_FILE)
    # Ensure only necessary columns are kept
    id_cols = ['item_id', 'state_id']
    feature_cols = ['ADI', 'CV2']
    features_df = features_df[id_cols + feature_cols]
except FileNotFoundError:
    print(f"\nERROR: Features file not found. Ensure 'dataprep.py' ran successfully.")
    exit()
print(f"Loaded features for {len(features_df)} target series.")

# --- 5. On-line Phase: Predict Costs and Find Optimal Policy ---

optimal_policies = []
print(f"\nOptimizing policies for {len(features_df)} series...")

# Iterate through each unique series (item_id, state_id)
for index, series_features in tqdm(features_df.iterrows(), total=len(features_df), desc="Optimizing"):
    
    # Extract ADI and CV2 for the current series
    series_input_features = series_features[feature_cols].values.reshape(1, -1) # Reshape for prediction
    adi = series_features['ADI']
    cv2 = series_features['CV2']

    # Create a DataFrame combining series features with all policy hparams
    # We need features for prediction: ADI, CV2, R, L, TSL
    prediction_df = HPARAMS_DF.copy()
    prediction_df['ADI'] = adi
    prediction_df['CV2'] = cv2
    prediction_df = prediction_df[['ADI', 'CV2', 'R', 'L', 'TSL']] # Ensure correct order

    # Predict cost elements for all 9000 (or 75) policies using the trained models (Step 7)
    predicted_costs = {}
    for target, model in models.items():
        predicted_costs[target] = model.predict(prediction_df)
    
    predicted_costs_df = pd.DataFrame(predicted_costs)
    
    # Estimate Total Inventory Cost for each policy (Step 8)
    # Using Equation 8 from the paper (simplified without price p_i)
    # C_Tot = C_H + C_LS + C_O
    # C_H ≈ h * AvgInventory * SimDays (Using daily cost)
    # C_LS ≈ b * LostSales
    # C_O = k * NumOrders
    
    # Note: Using predicted values directly. Paper uses p_i with h and b.
    # We apply cost parameters directly to the counts.
    predicted_costs_df['TotalCost'] = (
        h_daily * predicted_costs_df['AvgInventory'] * 365 + # Holding cost (adjust h for days)
        b * predicted_costs_df['LostSales'] +              # Shortage cost
        k * predicted_costs_df['NumOrders']                 # Ordering cost
    )
    
    # Combine predictions with policy parameters
    results_df = pd.concat([HPARAMS_DF, predicted_costs_df], axis=1)

    # Identify Optimal Hyperparameters (Step 9)
    # Find the row (policy) with the minimum predicted TotalCost
    optimal_row = results_df.loc[results_df['TotalCost'].idxmin()]

    # Store the optimal policy for this series
    optimal_policies.append({
        'item_id': series_features['item_id'],
        'state_id': series_features['state_id'],
        'ADI': adi,
        'CV2': cv2,
        'Optimal_R': optimal_row['R'],
        'Optimal_L': optimal_row['L'],
        'Optimal_TSL': optimal_row['TSL'],
        'Predicted_Min_Cost': optimal_row['TotalCost']
    })

# --- 6. Save Optimal Policies ---
optimal_policies_df = pd.DataFrame(optimal_policies)
optimal_policies_df.to_csv(OPTIMAL_PARAMS_FILE, index=False)

print("\nOptimization complete!")
print(f"Optimal inventory policies saved to: {OPTIMAL_PARAMS_FILE}")

# --- Next Steps ---
print("\n--- Next Steps (Phase 3, Steps 10 & 11) ---")
print("10. Evaluate Performance: Run final simulations using the optimal policies found.")
print("11. Compare with Benchmarks: Implement TO and FSH benchmarks for comparison.")
print("The script name for the next step could be: evaluation.py")


!!! Using reduced hyperparameter set (75 sets) for optimization !!!
Generated 75 policy hyperparameter sets for evaluation.
Using Cost Parameters: h=0.01, b=0.25, k=0.5

Loading trained LightGBM models...
  Loaded model for AvgInventory.
  Loaded model for LostSales.
  Loaded model for NumOrders.

Loading features for target series from: /kaggle/working/m5_demand_features.csv
Loaded features for 9147 target series.

Optimizing policies for 9147 series...


Optimizing: 100%|██████████| 9147/9147 [01:59<00:00, 76.23it/s]


Optimization complete!
Optimal inventory policies saved to: /kaggle/working/optimal_inventory_params.csv

--- Next Steps (Phase 3, Steps 10 & 11) ---
10. Evaluate Performance: Run final simulations using the optimal policies found.
11. Compare with Benchmarks: Implement TO and FSH benchmarks for comparison.
The script name for the next step could be: evaluation.py


In [9]:
import os
import pandas as pd
import numpy as np
from scipy.stats import norm
from tqdm import tqdm
import joblib # Potentially needed if reusing functions, though simulation logic is copied

# --- 1. Configuration and File Paths ---
# INPUT/OUTPUT directory (Must be /kaggle/working)
OUTPUT_DIR = '/kaggle/working'

# Input files
OPTIMAL_PARAMS_FILE = os.path.join(OUTPUT_DIR, 'optimal_inventory_params.csv')
LEVEL_11_SALES_FILE = os.path.join(OUTPUT_DIR, 'sales_train_level_11.csv') # Need full history for TO
# If using reduced hparams for training, load that simulation output for TO
SIMULATION_RESULTS_FILE = os.path.join(OUTPUT_DIR, 'ml_training_data_75_hparams.csv') 
CALENDAR_FILE = os.path.join(OUTPUT_DIR, 'calendar.csv') # Needed for dates
PRICES_FILE = os.path.join(OUTPUT_DIR, 'sell_prices.csv') # Needed for cost calculation

# Output file
EVALUATION_RESULTS_FILE = os.path.join(OUTPUT_DIR, 'evaluation_results.csv')

# --- 2. Evaluation Period and Cost Parameters ---
# Define the evaluation period (e.g., the second year, days 1914-1941 or similar)
# For M5, the final evaluation is d_1942 to d_1969 (28 days)
# For the paper's setup (1 year historical, 1 year evaluation), we need the next 365 days
# Let's assume evaluation uses days after TRAIN_DAYS (1913) up to the end of available data.
# We need the full sales file path from the *original* dataset location
INPUT_DIR_ORIGINAL = '/kaggle/input/m5-forecasting-accuracy'
SALES_FILE_FULL = os.path.join(INPUT_DIR_ORIGINAL, 'sales_train_evaluation.csv') # Use evaluation file if available

# Paper uses 1 year for evaluation (365 days)
EVAL_START_DAY = 1914 # Day after the training period ends
EVAL_DAYS = 28 # M5 competition final evaluation period
EVAL_DAY_COLS = [f'd_{i}' for i in range(EVAL_START_DAY, EVAL_START_DAY + EVAL_DAYS)]

# Inventory Cost Hyperparameters (h, b, k) from the paper (Section 3.4)
h = 0.01  # Annual holding cost %
b = 0.25  # Unit shortage cost %
k = 0.50  # Cost per order ($)
h_daily = h / 365 # Daily holding cost rate

# FSH Benchmark Parameters (Section 4.3)
FSH_PARAMS = {'R': 7, 'L': 3, 'TSL': 0.95}

# --- 3. Load Required Data ---
print("Loading necessary data for evaluation...")
try:
    optimal_policies_df = pd.read_csv(OPTIMAL_PARAMS_FILE)
    features_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'm5_demand_features.csv')) # ADI/CV2 needed
    
    # Load sales data for the evaluation period
    # Need item_id, state_id to merge with optimal policies
    # Usecols can optimize memory if needed
    all_sales_df = pd.read_csv(SALES_FILE_FULL) 
    
    # Check if EVAL_DAY_COLS exist in the loaded sales data
    missing_eval_cols = [col for col in EVAL_DAY_COLS if col not in all_sales_df.columns]
    if missing_eval_cols:
         print(f"Warning: Evaluation day columns missing from sales data: {missing_eval_cols}")
         # Attempt to load validation file if evaluation file is incomplete for the range
         all_sales_df = pd.read_csv(os.path.join(INPUT_DIR_ORIGINAL, 'sales_train_validation.csv'))
         missing_eval_cols = [col for col in EVAL_DAY_COLS if col not in all_sales_df.columns]
         if missing_eval_cols:
             print(f"ERROR: Cannot find evaluation period columns {EVAL_DAY_COLS} in sales data.")
             exit()

    # Aggregate full sales data to Level 11 for evaluation period
    print("Aggregating sales data to Level 11 for the evaluation period...")
    level_11_eval_sales = all_sales_df.groupby(['item_id', 'state_id'])[EVAL_DAY_COLS].sum().reset_index()

    # Load simulation results to determine the TO benchmark
    simulation_results_df = pd.read_csv(SIMULATION_RESULTS_FILE)
    
    # Load prices for accurate cost calculation (Optional but recommended)
    prices_df = pd.read_csv(PRICES_FILE)
    # Load calendar to map 'd' columns to 'wm_yr_wk' for price merging
    calendar_df = pd.read_csv(CALENDAR_FILE, usecols=['d', 'wm_yr_wk'])
    d_to_wk_map = calendar_df.set_index('d')['wm_yr_wk'].to_dict()

except FileNotFoundError as e:
    print(f"\nERROR: Could not load required file: {e}. Ensure previous steps ran successfully.")
    exit()

# --- 4. Implement True Optimal (TO) Benchmark (Step 11) ---
print("\nDetermining True Optimal (TO) benchmark policies...")

# Calculate Total Cost for each simulation run in the *historical* data
# Merge price data to simulation results for accurate cost calc (if desired)
# Simplified cost calculation (as used in optimization.py) for consistency:
simulation_results_df['HistoricalCost'] = (
    h_daily * simulation_results_df['AvgInventory'] * 365 + # Holding cost
    b * simulation_results_df['LostSales'] +              # Shortage cost
    k * simulation_results_df['NumOrders']                 # Ordering cost
)

# Find the best R, L, TSL for each series based on lowest historical cost
to_policies_idx = simulation_results_df.groupby(['item_id', 'state_id'])['HistoricalCost'].idxmin()
to_policies_df = simulation_results_df.loc[to_policies_idx, ['item_id', 'state_id', 'R', 'L', 'TSL']].reset_index(drop=True)
to_policies_df.rename(columns={'R': 'TO_R', 'L': 'TO_L', 'TSL': 'TO_TSL'}, inplace=True)
print(f"Determined TO policies for {len(to_policies_df)} series.")

# --- 5. Re-Implement Simulation Logic for Evaluation (Step 10) ---
# We reuse the core logic from simulation.py, but run it on EVAL_DAY_COLS

def calculate_policy_stats_eval(history): # Use historical part for stats
    D_hat = history.mean()
    sigma_D = history.std(ddof=1) if len(history) > 1 else 1.0
    return D_hat, sigma_D

def run_evaluation_simulation(historical_sales, eval_sales, R, L, TSL):
    """
    Runs the (R, s, S) simulation on the evaluation period sales data,
    using stats calculated from historical_sales.
    Returns realized cost elements and service level.
    """
    if len(historical_sales) == 0 or len(eval_sales) == 0:
        return np.nan, np.nan, np.nan, np.nan # Handle cases with insufficient data

    # 1. Calculate Policy Parameters using HISTORICAL data stats
    D_hat, sigma_D = calculate_policy_stats_eval(historical_sales)
    Z = norm.ppf(TSL)
    SS = Z * sigma_D * np.sqrt(R + L)
    policy_term = D_hat * (R + L)
    s_t = policy_term + SS
    S_t = policy_term + SS

    # Initialize simulation variables
    inventory_level = S_t # Start with target inventory level
    orders_placed = 0
    inventory_levels = []
    lost_sales_list = []
    total_demand_list = []

    # 2. Simulation Loop over EVALUATION period
    num_eval_days = len(eval_sales)
    for t in range(num_eval_days):
        order_quantity = 0
        if t % R == 0:
            if inventory_level <= s_t:
                order_quantity = S_t - inventory_level
                orders_placed += 1

        demand = eval_sales.iloc[t]
        total_demand_list.append(demand)
        satisfied_sales = min(inventory_level, demand)
        lost_sales = demand - satisfied_sales
        inventory_level -= satisfied_sales
        inventory_levels.append(inventory_level)
        lost_sales_list.append(lost_sales)

    # 3. Calculate Realized Performance Metrics
    avg_inventory_real = np.mean(inventory_levels)
    total_lost_sales_real = np.sum(lost_sales_list)
    total_orders_real = orders_placed
    
    # Calculate Realized Service Level (SL) = 1 - (Total Lost Sales / Total Demand)
    total_demand_real = np.sum(total_demand_list)
    realized_sl = (1 - (total_lost_sales_real / total_demand_real)) if total_demand_real > 0 else 1.0

    return avg_inventory_real, total_lost_sales_real, total_orders_real, realized_sl

# --- 6. Run Evaluations for MOIC, TO, FSH ---
print("\nRunning evaluation simulations...")

evaluation_results = []

# Merge optimal policies with evaluation sales data and historical sales data
# Need historical sales (e.g., last 365 days before eval) to calculate policy stats
hist_day_cols = [f'd_{i}' for i in range(EVAL_START_DAY - SIM_DAYS, EVAL_START_DAY)]
level_11_hist_sales = all_sales_df.groupby(['item_id', 'state_id'])[hist_day_cols].sum().reset_index()

eval_df = optimal_policies_df.merge(level_11_eval_sales, on=['item_id', 'state_id'], how='inner')
eval_df = eval_df.merge(level_11_hist_sales, on=['item_id', 'state_id'], how='inner')
eval_df = eval_df.merge(to_policies_df, on=['item_id', 'state_id'], how='inner')

# Iterate through each series to evaluate all policies
for index, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating Policies"):
    item_id = row['item_id']
    state_id = row['state_id']
    
    hist_sales = row[hist_day_cols]
    eval_sales = row[EVAL_DAY_COLS]

    results = {'item_id': item_id, 'state_id': state_id}

    # Evaluate MOIC Optimal Policy
    moic_r, moic_l, moic_tsl = row['Optimal_R'], row['Optimal_L'], row['Optimal_TSL']
    avg_inv, ls, orders, sl = run_evaluation_simulation(hist_sales, eval_sales, moic_r, moic_l, moic_tsl)
    results['MOIC_AvgInv'] = avg_inv
    results['MOIC_LS'] = ls
    results['MOIC_Orders'] = orders
    results['MOIC_SL'] = sl
    results['MOIC_Cost'] = h_daily * avg_inv * EVAL_DAYS + b * ls + k * orders

    # Evaluate TO Benchmark Policy
    to_r, to_l, to_tsl = row['TO_R'], row['TO_L'], row['TO_TSL']
    avg_inv, ls, orders, sl = run_evaluation_simulation(hist_sales, eval_sales, to_r, to_l, to_tsl)
    results['TO_AvgInv'] = avg_inv
    results['TO_LS'] = ls
    results['TO_Orders'] = orders
    results['TO_SL'] = sl
    results['TO_Cost'] = h_daily * avg_inv * EVAL_DAYS + b * ls + k * orders

    # Evaluate FSH Benchmark Policy
    fsh_r, fsh_l, fsh_tsl = FSH_PARAMS['R'], FSH_PARAMS['L'], FSH_PARAMS['TSL']
    avg_inv, ls, orders, sl = run_evaluation_simulation(hist_sales, eval_sales, fsh_r, fsh_l, fsh_tsl)
    results['FSH_AvgInv'] = avg_inv
    results['FSH_LS'] = ls
    results['FSH_Orders'] = orders
    results['FSH_SL'] = sl
    results['FSH_Cost'] = h_daily * avg_inv * EVAL_DAYS + b * ls + k * orders
    
    evaluation_results.append(results)

# --- 7. Save Final Evaluation Results ---
evaluation_results_df = pd.DataFrame(evaluation_results)
evaluation_results_df.to_csv(EVALUATION_RESULTS_FILE, index=False)

print("\nEvaluation complete!")
print(f"Final evaluation results saved to: {EVALUATION_RESULTS_FILE}")

# --- Final Step ---
print("\n--- Next Step (Phase 4, Step 12) ---")
print("12. Analyze and Summarize Results: Compare the performance metrics (Cost, SL) across MOIC, TO, and FSH.")
print("   (Typically done in a separate analysis script or notebook)")


Loading necessary data for evaluation...
Aggregating sales data to Level 11 for the evaluation period...

Determining True Optimal (TO) benchmark policies...
Determined TO policies for 9147 series.

Running evaluation simulations...


Evaluating Policies: 100%|██████████| 9147/9147 [00:30<00:00, 304.54it/s]



Evaluation complete!
Final evaluation results saved to: /kaggle/working/evaluation_results.csv

--- Next Step (Phase 4, Step 12) ---
12. Analyze and Summarize Results: Compare the performance metrics (Cost, SL) across MOIC, TO, and FSH.
   (Typically done in a separate analysis script or notebook)
